# TxGNN/Jouvence KG — deep schema inspection

Notebook interactif pour inspecter le schema TxGNN/Jouvence en détail :

- node types et ontologies primaires
- relations, `RelationKind`, directness, lifecycle status (`ACTIVE`, `DERIVED`, `LEGACY_INDEX`, `DEPRECATED`)
- relations candidates hors schema canonical
- couverture réelle du KG canonical GCS (`nodes/`, `edges/`, `evidence/`)
- audit evidence relation par relation
- inspection détaillée d'une relation sélectionnée

Par défaut, le notebook lit le KG canonical monté localement :

```text
/mnt/gcs/jouvencekb/kg/v2
```

Il ne modifie rien : toutes les cellules sont read-only.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq

# Works when executed from repo root or from notebooks/.
REPO_ROOT = Path.cwd()
if (REPO_ROOT / "manage_db").exists():
    sys.path.insert(0, str(REPO_ROOT))
elif (REPO_ROOT.parent / "manage_db").exists():
    sys.path.insert(0, str(REPO_ROOT.parent))
else:
    raise RuntimeError(f"Could not locate repo root from {Path.cwd()}")

from manage_db import kg_evidence, kg_storage
from manage_db.audit_edge_evidence import audit_edge_evidence
from manage_db.audit_kg_coverage import audit_coverage
from manage_db.kg_schema import (
    CANDIDATE_RELATIONS,
    NODE_TYPES,
    RELATIONS,
    RELATION_BY_NAME,
    RelationStatus,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 240)

KG_ROOT = Path(os.environ.get("TXGNN_KG_ROOT", "/mnt/gcs/jouvencekb/kg/v2"))
root = kg_storage.open_kg_root(str(KG_ROOT))
KG_ROOT

## 1. Canonical KG coverage snapshot

Reads Parquet metadata only for row counts; no full table scans.

In [ ]:
coverage = audit_coverage(KG_ROOT)
coverage_summary = pd.DataFrame([
    {"kind": "node_files", "present": len(coverage.node_counts), "expected": len(NODE_TYPES), "rows": coverage.total_nodes},
    {"kind": "edge_files", "present": len(coverage.edge_counts), "expected": len(RELATIONS), "rows": coverage.total_edges},
    {"kind": "missing_nodes", "present": 0, "expected": len(coverage.missing_nodes), "rows": None},
    {"kind": "missing_edges", "present": 0, "expected": len(coverage.missing_edges), "rows": None},
])
coverage_summary

In [ ]:
print("Missing node files:", coverage.missing_nodes)
print("Missing edge files:")
for name in coverage.missing_edges:
    print(" -", name)

## 2. Node schema

`NODE_TYPES` is the source of truth for node type, primary ontology, example ID, and xref columns.

In [ ]:
node_rows = []
for node_type, info in NODE_TYPES.items():
    node_rows.append({
        "node_type": node_type.value,
        "primary_ontology": info.primary_ontology,
        "id_format": info.id_format,
        "example_id": info.example_id,
        "bionty_registry": info.bionty_registry or "",
        "xref_columns": ", ".join(info.xref_columns),
        "gcs_rows": coverage.node_counts.get(node_type.value),
        "gcs_present": node_type.value in coverage.node_counts,
    })
node_df = pd.DataFrame(node_rows).sort_values("node_type")
node_df

## 3. Relation schema + canonical edge/evidence counts

This table merges `kg_schema.RELATIONS` with actual canonical Parquet row counts and evidence row counts.

In [ ]:
evidence_counts = {}
for relation in kg_evidence.list_evidence(root):
    path = root._join("evidence", f"{relation}.parquet")
    evidence_counts[relation] = pq.ParquetFile(path, filesystem=root.fs).metadata.num_rows

relation_rows = []
for rel in RELATIONS:
    relation_rows.append({
        "relation": rel.name,
        "source": rel.source.value,
        "target": rel.target.value,
        "kind": rel.kind.value,
        "direct": rel.direct,
        "status": rel.status.value,
        "replacement": rel.replacement,
        "notes": rel.notes,
        "edge_rows": coverage.edge_counts.get(rel.name),
        "edge_present": rel.name in coverage.edge_counts,
        "evidence_rows": evidence_counts.get(rel.name, 0),
        "evidence_present": rel.name in evidence_counts,
    })
relation_df = pd.DataFrame(relation_rows)
relation_df.sort_values(["status", "kind", "relation"])

In [ ]:
# Compact dashboard: canonical edges that still have no evidence layer.
edge_without_evidence = relation_df[(relation_df.edge_present) & (~relation_df.evidence_present)].copy()
edge_without_evidence[["relation", "source", "target", "kind", "status", "edge_rows", "notes"]].sort_values("edge_rows", ascending=False)

## 4. Lifecycle cleanup views

Useful for reviewing `TODEL`, legacy-index, and derived-edge decisions.

In [ ]:
for status in RelationStatus:
    subset = relation_df[relation_df.status == status.value]
    print(f"{status.value}: {len(subset)}")

relation_df[relation_df.status != RelationStatus.ACTIVE.value][
    ["relation", "source", "target", "kind", "status", "replacement", "edge_rows", "evidence_rows", "notes"]
].sort_values(["status", "relation"])

In [ ]:
candidate_df = pd.DataFrame([
    {
        "candidate_relation": rel.name,
        "source": rel.source.value,
        "target": rel.target.value,
        "kind": rel.kind.value,
        "direct": rel.direct,
        "recommendation": rel.recommendation,
    }
    for rel in CANDIDATE_RELATIONS
])
candidate_df

## 5. Evidence audit

Runs the canonical evidence audit for all evidence files currently present.

In [ ]:
evidence_relations = kg_evidence.list_evidence(root)
print("Evidence files:", evidence_relations)

evidence_audit = audit_edge_evidence(KG_ROOT, relations=evidence_relations)
evidence_audit_df = pd.DataFrame([
    {
        "relation": name,
        "edge_rows": report.edge_rows,
        "evidence_rows": report.evidence_rows,
        "edges_without_evidence": report.edges_without_evidence,
        "evidence_without_edge": report.evidence_without_edge,
        "ok": report.ok,
    }
    for name, report in evidence_audit.relation_reports.items()
]).sort_values("relation")
evidence_audit_df

In [ ]:
# Evidence support composition.
composition = []
for relation in evidence_relations:
    ev = kg_evidence.read_evidence(root, relation, columns=["evidence_type", "source", "source_dataset", "predicate"])
    grouped = ev.groupby(["evidence_type", "source", "source_dataset", "predicate"], dropna=False).size().reset_index(name="rows")
    grouped.insert(0, "relation", relation)
    composition.append(grouped)

evidence_composition_df = pd.concat(composition, ignore_index=True) if composition else pd.DataFrame()
evidence_composition_df.sort_values(["relation", "rows"], ascending=[True, False]).head(100)

## 6. Inspect one relation in detail

Change `RELATION` below and re-run the following cells.

In [ ]:
RELATION = "mutation_affects_molecule_response"  # change me
rel = RELATION_BY_NAME[RELATION]
print(rel)
print("edge rows:", coverage.edge_counts.get(RELATION))
print("evidence rows:", evidence_counts.get(RELATION, 0))

In [ ]:
# Edge Parquet schema and small sample.
edge_path = root._join("edges", f"{RELATION}.parquet")
edge_pf = pq.ParquetFile(edge_path, filesystem=root.fs)
print(edge_pf.schema)
edge_sample = pq.read_table(edge_path, filesystem=root.fs).slice(0, 10).to_pandas()
edge_sample

In [ ]:
# Evidence Parquet schema and sample, if present.
evidence_path = root._join("evidence", f"{RELATION}.parquet")
if root.fs.exists(evidence_path):
    ev_pf = pq.ParquetFile(evidence_path, filesystem=root.fs)
    print(ev_pf.schema)
    evidence_sample = pq.read_table(evidence_path, filesystem=root.fs).slice(0, 20).to_pandas()
    display(evidence_sample)
else:
    print(f"No evidence file for {RELATION}")

In [ ]:
# Per-column quick stats for the selected edge file.
# This reads one selected edge file fully; for huge relations, change RELATION first if needed.
edge_full_for_stats = pq.read_table(edge_path, filesystem=root.fs).to_pandas()
summary_rows = []
for col in edge_full_for_stats.columns:
    s = edge_full_for_stats[col]
    examples = s.dropna().astype(str).unique()[:5]
    summary_rows.append({
        "column": col,
        "dtype": str(s.dtype),
        "non_null": int(s.notna().sum()),
        "null": int(s.isna().sum()),
        "nunique": int(s.astype(str).nunique(dropna=True)),
        "examples": list(examples),
    })
pd.DataFrame(summary_rows)

## 7. Endpoint type sanity checks

Highlights relations where the declared schema target/source does not match sampled canonical edge `x_type/y_type` values. This is useful for legacy gene/protein-conflated edges.

In [ ]:
type_checks = []
for rel in RELATIONS:
    if rel.name not in coverage.edge_counts:
        continue
    path = root._join("edges", f"{rel.name}.parquet")
    pf = pq.ParquetFile(path, filesystem=root.fs)
    try:
        batch = next(pf.iter_batches(batch_size=5000, columns=["x_type", "y_type"]))
        sample = batch.to_pandas()
    except StopIteration:
        sample = pd.DataFrame(columns=["x_type", "y_type"])
    x_types = sorted(sample["x_type"].dropna().astype(str).unique())
    y_types = sorted(sample["y_type"].dropna().astype(str).unique())
    type_checks.append({
        "relation": rel.name,
        "declared_source": rel.source.value,
        "sample_x_types": ", ".join(x_types),
        "source_match": x_types == [rel.source.value],
        "declared_target": rel.target.value,
        "sample_y_types": ", ".join(y_types),
        "target_match": y_types == [rel.target.value],
        "edge_rows": coverage.edge_counts.get(rel.name),
    })
type_check_df = pd.DataFrame(type_checks)
type_check_df[(~type_check_df.source_match) | (~type_check_df.target_match)].sort_values("edge_rows", ascending=False)

## 8. Export tables for review

Optional: uncomment to write CSV summaries under `docs/`.

In [ ]:
# Uncomment if you want materialized CSV snapshots.
# out_dir = Path("docs/schema_inspection_exports")
# out_dir.mkdir(parents=True, exist_ok=True)
# node_df.to_csv(out_dir / "nodes.csv", index=False)
# relation_df.to_csv(out_dir / "relations.csv", index=False)
# evidence_audit_df.to_csv(out_dir / "evidence_audit.csv", index=False)
# evidence_composition_df.to_csv(out_dir / "evidence_composition.csv", index=False)
# type_check_df.to_csv(out_dir / "endpoint_type_checks.csv", index=False)
# out_dir